In [2]:
!pip install yfinance 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [yfinance]3/4 [yfinance]


In [ ]:
import yfinance as yf
import pandas as pd
from scipy import stats
import numpy as np
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
# extracting all the tickers directly from spf500
sp500 = pd.read_html(StringIO(html))[0]
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()



# downloading the price per month from yahoo finance
prices = yf.download(tickers, start = "2020-01-01", interval = "1mo")["Close"]
returns = prices.pct_change()


def run_strategy(prices,returns, j, k):

    """ Running the momentum strategy for different j and k combinationsa and ranking stocks every month 
    where we long the top 20% and short the bottom 20%. Hold that portfolio for k months with overlapping 
    portfolios. Returns the monthly long-short return portfolAios """


    # returns for the trailing j period and shifting up one month to prevent immediate spike effect
    momentum_j = prices.pct_change(periods = j)
    signal = momentum_j.shift(1)

    # ranking the stocks based on the returns(highest to lowest)
    ranks = signal.rank(axis = 1, ascending = False)
    n = signal.notna().sum(axis = 1)
    # top 20% of n as winners and bottom 20% of n as losers
    winners = ranks.le(n * 0.2, axis = 0)
    losers = ranks.gt(n * 0.8, axis = 0)
    # weight(+1/-1), whether we are going long or shorting
    weight = winners.astype(int) - losers.astype(int)

    # layering the weight over k months
    layers = [weight.shift(i) for i in range(1, k + 1)]
    # the return for the corresponding j and k protfolio
    avg_weight = sum(layers) / k
    port_return = (avg_weight * returns).sum(axis = 1)
    return port_return.dropna()


""" the 16x16 grid for the mean return and the t-stat(hypothesis testing) for 
every j and k combination """
results = []
for j in [3, 6, 9, 12]:
    for k in [3, 6, 9, 12]:
        r = run_strategy(prices, returns, j, k)
        avg_mean = r.mean()
        t_stat = stats.ttest_1samp(r, 0).statistic
        results.append({"J": j, "K": k,
                        "mean_monthly": round(avg_mean, 4),
                        "t_stat": round(t_stat, 2)})

grid = pd.DataFrame(results)
grid


[***************       31%                       ]  158 of 503 completed